# IC-GAN Batch Image Generator

This Colab Notebook provides the code to generate images with IC-GAN (Instance-Conditioned GAN). This notebook is based from a Colab Notebook image generator from https://github.com/facebookresearch/ic_gan which has since been not working out of the box because of its age (5 years). The code has been modified to be able to process multiple images from a folder in Google Drive and export back the generated GAN images to a folder in Google Drive.

In [1]:
# Use older PyTorch for this 5 year old code

# 1. Uninstall the current incompatible versions
!pip uninstall -y torch torchvision torchaudio

# 2. Install PyTorch 2.5.1 with CUDA 12.1 support (Standard for Colab T4)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 71.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66

In [4]:
#@title Verify Torch Version
import torch
import torchvision

print(f"Torch: {torch.__version__}")
print(f"Vision: {torchvision.__version__}")
print(f"CUDA status: {torch.cuda.is_available()}")

Torch: 2.5.1+cu121
Vision: 0.20.1+cu121
CUDA status: True


In [5]:
#@title Restart after running this cell!

!nvidia-smi -L

import subprocess

CUDA_version = [s for s in subprocess.check_output(["nvcc", "--version"]).decode("UTF-8").split(", ") if s.startswith("release")][0].split(" ")[-1]
print("CUDA version:", CUDA_version)

if CUDA_version == "10.1":
    torch_version_suffix = "+cu101"
elif CUDA_version == "10.2":
    torch_version_suffix = "+cu102"
else:
    torch_version_suffix = "+cu111"

GPU 0: Tesla T4 (UUID: GPU-8f51c59b-acf5-7c11-89e4-61c0b3e66b0a)
CUDA version: 12.8


In [6]:
#@title Setup and Download Libraries
!git clone https://github.com/facebookresearch/ic_gan.git

%cd /content
# Uncompress required files
!wget https://dl.fbaipublicfiles.com/ic_gan/cc_icgan_biggan_imagenet_res256.tar.gz
!tar -xvf cc_icgan_biggan_imagenet_res256.tar.gz
!wget https://dl.fbaipublicfiles.com/ic_gan/icgan_biggan_imagenet_res256.tar.gz
!tar -xvf icgan_biggan_imagenet_res256.tar.gz
!wget https://dl.fbaipublicfiles.com/ic_gan/stored_instances.tar.gz
!tar -xvf stored_instances.tar.gz

!pip install pytorch-pretrained-biggan

# !git clone --depth 1 https://github.com/openai/CLIP
!pip install ftfy
# %cd /content/CLIP
# import clip
# last_clip_model = 'ViT-B/32'
# perceptor, preprocess = clip.load(last_clip_model)

import nltk
nltk.download('wordnet')

!pip install cma


Cloning into 'ic_gan'...
remote: Enumerating objects: 242, done.
remote: Total 242 (delta 0), reused 0 (delta 0), pack-reused 242 (from 1)
Receiving objects: 100% (242/242), 7.10 MiB | 14.69 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content
--2026-04-02 21:05:32--  https://dl.fbaipublicfiles.com/ic_gan/cc_icgan_biggan_imagenet_res256.tar.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.81, 13.249.182.62, 13.249.182.39, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.81|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 342709323 (327M) [application/gzip]
Saving to: ‘cc_icgan_biggan_imagenet_res256.tar.gz’

cc_icgan_biggan_ima 100%[===================>] 326.83M   201MB/s    in 1.6s    

2026-04-02 21:05:34 (201 MB/s) - ‘cc_icgan_biggan_imagenet_res256.tar.gz’ saved [342709323/342709323]

cc_icgan_biggan_imagenet_res256/
cc_icgan_biggan_imagenet_res256/state_dict_best0.pth
cc_icgan_biggan_imagenet_res

[nltk_data] Downloading package wordnet to /root/nltk_data...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 9.9 MB/s eta 0:00:00


In [7]:
#@title Fix missing libary used by resnet.py from IC_GAN repo

!sed -i 's/from torchvision.models.utils import load_state_dict_from_url/from torch.hub import load_state_dict_from_url/g' /content/ic_gan/data_utils/resnet.py

In [8]:
#@title Prepare functions
from pytorch_pretrained_biggan import BigGAN, convert_to_images, one_hot_from_names, utils

%cd /content/ic_gan/
import sys
import os
sys.path[0] = '/content/ic_gan/inference'
sys.path.insert(1, os.path.join(sys.path[0], ".."))
import torch

import numpy as np
import torch
import torchvision
import sys
torch.manual_seed(np.random.randint(sys.maxsize))
import imageio
from IPython.display import HTML, Image, clear_output
from PIL import Image as Image_PIL
from scipy.stats import truncnorm, dirichlet
from torch import nn
from nltk.corpus import wordnet as wn
from base64 import b64encode
from time import time
import cma
from cma.sigma_adaptation import CMAAdaptSigmaCSA, CMAAdaptSigmaTPA
import warnings
warnings.simplefilter("ignore", cma.evolution_strategy.InjectionWarning)
import torchvision.transforms as transforms
import inference.utils as inference_utils
import data_utils.utils as data_utils
from BigGAN_PyTorch.BigGAN import Generator as generator
import sklearn.metrics

def replace_to_inplace_relu(model): #saves memory; from https://github.com/minyoungg/pix2latent/blob/master/pix2latent/model/biggan.py
    for child_name, child in model.named_children():
        if isinstance(child, nn.ReLU):
            setattr(model, child_name, nn.ReLU(inplace=False))
        else:
            replace_to_inplace_relu(child)
    return

def save(out,name=None, torch_format=True):
  if torch_format:
    with torch.no_grad():
      out = out.cpu().numpy()
  img = convert_to_images(out)[0]
  if name:
    imageio.imwrite(name, np.asarray(img))
  return img

hist = []
def checkin(i, best_ind, total_losses, losses, regs, out, noise=None, emb=None, probs=None):
  global sample_num, hist
  name = None
  if save_every and i%save_every==0:
    name = '/content/output/frame_%05d.jpg'%sample_num
  pil_image = save(out, name)
  vals0 = [sample_num, i, total_losses[best_ind], losses[best_ind], regs[best_ind], np.mean(total_losses), np.mean(losses), np.mean(regs), np.std(total_losses), np.std(losses), np.std(regs)]
  stats = 'sample=%d iter=%d best: total=%.2f cos=%.2f reg=%.3f avg: total=%.2f cos=%.2f reg=%.3f std: total=%.2f cos=%.2f reg=%.3f'%tuple(vals0)
  vals1 = []
  if noise is not None:
    vals1 = [np.mean(noise), np.std(noise)]
    stats += ' noise: avg=%.2f std=%.3f'%tuple(vals1)
  vals2 = []
  if emb is not None:
    vals2 = [emb.mean(),emb.std()]
    stats += ' emb: avg=%.2f std=%.3f'%tuple(vals2)
  elif probs:
    best = probs[best_ind]
    inds = np.argsort(best)[::-1]
    probs = np.array(probs)
    vals2 = [ind2name[inds[0]], best[inds[0]], ind2name[inds[1]], best[inds[1]], ind2name[inds[2]], best[inds[2]], np.sum(probs >= 0.5)/pop_size,np.sum(probs >= 0.3)/pop_size,np.sum(probs >= 0.1)/pop_size]
    stats += ' 1st=%s(%.2f) 2nd=%s(%.2f) 3rd=%s(%.2f) components: >=0.5:%.0f, >=0.3:%.0f, >=0.1:%.0f'%tuple(vals2)
  hist.append(vals0+vals1+vals2)
  if show_every and i%show_every==0:
    clear_output()
    display(pil_image)
  print(stats)
  sample_num += 1

def load_icgan(experiment_name, root_ = '/content'):
  root = os.path.join(root_, experiment_name)
  config = torch.load("%s/%s.pth" %
                      (root, "state_dict_best0"))['config']

  config["weights_root"] = root_
  config["model_backbone"] = 'biggan'
  config["experiment_name"] = experiment_name
  G, config = inference_utils.load_model_inference(config)
  G.cuda()
  G.eval()
  return G

def get_output(noise_vector, input_label, input_features):
  if stochastic_truncation: #https://arxiv.org/abs/1702.04782
    with torch.no_grad():
      trunc_indices = noise_vector.abs() > 2*truncation
      size = torch.count_nonzero(trunc_indices).cpu().numpy()
      trunc = truncnorm.rvs(-2*truncation, 2*truncation, size=(1,size)).astype(np.float32)
      noise_vector.data[trunc_indices] = torch.tensor(trunc, requires_grad=requires_grad, device='cuda')
  else:
    noise_vector = noise_vector.clamp(-2*truncation, 2*truncation)
  if input_label is not None:
    input_label = torch.LongTensor(input_label)
  else:
    input_label = None

  out = model(noise_vector, input_label.cuda() if input_label is not None else None, input_features.cuda() if input_features is not None else None)

  if channels==1:
    out = out.mean(dim=1, keepdim=True)
    out = out.repeat(1,3,1,1)
  return out

def normality_loss(vec): #https://arxiv.org/abs/1903.00925
    mu2 = vec.mean().square()
    sigma2 = vec.var()
    return mu2+sigma2-torch.log(sigma2)-1


def load_generative_model(gen_model, last_gen_model, experiment_name, model):
  # Load generative model
  if gen_model != last_gen_model:
    model = load_icgan(experiment_name, root_ = '/content')
    last_gen_model = gen_model
  return model, last_gen_model

def load_feature_extractor(gen_model, last_feature_extractor, feature_extractor):
  # Load feature extractor to obtain instance features
  feat_ext_name = 'classification' if gen_model == 'cc_icgan' else 'selfsupervised'
  if last_feature_extractor != feat_ext_name:
    if feat_ext_name == 'classification':
      feat_ext_path = ''
    else:
      !curl -L -o /content/swav_pretrained.pth.tar -C - 'https://dl.fbaipublicfiles.com/deepcluster/swav_800ep_pretrain.pth.tar'
      feat_ext_path = '/content/swav_pretrained.pth.tar'
    last_feature_extractor = feat_ext_name
    feature_extractor = data_utils.load_pretrained_feature_extractor(feat_ext_path, feature_extractor = feat_ext_name)
    feature_extractor.eval()
  return feature_extractor, last_feature_extractor

norm_mean = torch.Tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
norm_std = torch.Tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def preprocess_input_image(input_image_path, size):
  pil_image = Image_PIL.open(input_image_path).convert('RGB')
  transform_list =  transforms.Compose([data_utils.CenterCropLongEdge(), transforms.Resize((size,size)), transforms.ToTensor(), transforms.Normalize(norm_mean, norm_std)])
  tensor_image = transform_list(pil_image)
  tensor_image = torch.nn.functional.interpolate(tensor_image.unsqueeze(0), 224, mode="bicubic", align_corners=True)
  return tensor_image

def preprocess_generated_image(image):
  transform_list =  transforms.Normalize(norm_mean, norm_std)
  image = transform_list(image*0.5 + 0.5)
  image = torch.nn.functional.interpolate(image, 224, mode="bicubic", align_corners=True)
  return image

last_gen_model = None
last_feature_extractor = None
model = None
feature_extractor = None

/content/ic_gan
Faiss library not found!


In [10]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
#@title Load Input Images where GAN images will be based upon

import shutil
import os
from tqdm.notebook import tqdm

destination_folder = '/content/image_input'

if os.path.exists(destination_folder):
    shutil.rmtree(destination_folder)
    print(f"Directory {destination_folder} has been removed.")
else:
    print(f"Directory {destination_folder} does not exist.")

# Where input image files are currently located
source_folder = '/content/drive/MyDrive/machine_learning/input_images'

# Create the destination folder if it doesn't exist yet
# os.makedirs(destination_folder, exist_ok=True)

# Get list of all files to calculate total progress
files_to_copy = []
for root, dirs, files in os.walk(source_folder):
    for file in files:
        files_to_copy.append(os.path.join(root, file))

# Execution with visual loader
print(f"Transferring {len(files_to_copy)} files...")

for file_path in tqdm(files_to_copy, desc="Downloading from Drive", unit="file"):
    # Determine the relative path to maintain folder structure
    rel_path = os.path.relpath(file_path, source_folder)
    dest_file_path = os.path.join(destination_folder, rel_path)

    # Ensure subfolders exist
    os.makedirs(os.path.dirname(dest_file_path), exist_ok=True)

    # Copy the file
    shutil.copy2(file_path, dest_file_path)

print("\n✅ Transfer Complete!")

Directory /content/image_input has been removed.
Transferring 5 files...



✅ Transfer Complete!


## Initiate Generator parameters

In [13]:
import os
from pathlib import Path
import numpy as np
import torch

gen_model = 'icgan'
if gen_model == 'icgan':
  experiment_name = 'icgan_biggan_imagenet_res256'
else:
  experiment_name = 'cc_icgan_biggan_imagenet_res256'
#last_gen_model = experiment_name
size = '256'
input_feature_index =   3
class_index =   538
num_samples_ranked =   16
num_samples_total =    160
truncation =  0.7
stochastic_truncation = False
download_file = False
seed =  50
if seed == 0:
  seed = None
noise_size = 128
class_size = 1000
channels = 3
batch_size = 4
if gen_model == 'icgan':
  class_index = None
if 'biggan' in gen_model:
  input_feature_index = None

# Re-initialize global model/feature extractor variables to ensure clean state
# global last_gen_model, last_feature_extractor, model, feature_extractor
last_gen_model = None
last_feature_extractor = None
model = None
feature_extractor = None

# --- NEW CONFIGURATION FOR BATCH PROCESSING ---
input_folder = "/content/image_input" ##param {type:"string"}
results_output = "/content/icgan_results"
os.makedirs(results_output, exist_ok=True)

# Get list of valid images in the folder
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.JPEG')
image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_extensions)]

print(f"Found {len(image_files)} images to process.")



Found 5 images to process.


## Load Generative Model

In [14]:
# Ensure seed is set for reproducibility for noise generation
state = None if not seed else np.random.RandomState(seed)
np.random.seed(seed)

# Determine feature extractor name based on gen_model
feature_extractor_name = 'classification' if gen_model == 'cc_icgan' else 'selfsupervised'

# Load feature extractor
feature_extractor, last_feature_extractor = load_feature_extractor(gen_model, last_feature_extractor, feature_extractor)

# Load generative model
model, last_gen_model = load_generative_model(gen_model, last_gen_model, experiment_name, model)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  108M  100  108M    0     0   178M      0 --:--:-- --:--:-- --:--:--  178M
using resnet50 to extract features
Loading pretrained weights from:  /content/swav_pretrained.pth.tar


/content/ic_gan/inference/../data_utils/utils.py:322: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_path)


key  projection_head.0.weight  not in dict
key  projection_head.0.bias  not in dict
key  projection_head.1.weight  not in dict
key  projection_head.1.bias  not in dict
key  projection_head.1.running_mean  not in dict
key  projection_head.1.running_var  not in dict
key  projection_head.1.num_batches_tracked  not in dict
key  projection_head.3.weight  not in dict
key  projection_head.3.bias  not in dict
key  prototypes.weight  not in dict
Network key  fc.weight  not in dict to load
Network key  fc.bias  not in dict to load
For name best  best0  we have an FID:  22.453704833984375
Checkpoint with name  best1  not in folder.
Final name selected is  best0
Loading best0 weights from /content/icgan_biggan_imagenet_res256...
Experiment name is icgan_biggan_imagenet_res256


/tmp/ipykernel_7347/2477983317.py:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  config = torch.load("%s/%s.pth" %
/content/ic_gan/inference/../inference/utils.py:292: Fu

Adding attention layer in G at resolution 64
Param count for Gs initialized parameters: 90014147
Loading weights...
Loading best0 weights from /content/icgan_biggan_imagenet_res256...


/content/ic_gan/inference/../BigGAN_PyTorch/utils.py:1260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Putting G in eval mode..


In [17]:
# @title Apply ReLU inplace replacement for memory saving, Prepare Noise Vector, Input Label where GAN images will be based upon
# Apply ReLU inplace replacement for memory saving
replace_to_inplace_relu(model)

# Create noise vector once before the loop
noise_vector = truncnorm.rvs(-2*truncation, 2*truncation, size=(num_samples_total, noise_size), random_state=state).astype(np.float32)
noise_vector = torch.tensor(noise_vector, requires_grad=False, device='cuda')

# Create input_label once before the loop (if applicable)
if gen_model == 'cc_icgan' and class_index is not None:
  # Assuming ind2name is globally available from cell fa6c7629
  print('Conditioning on class: ', ind2name[class_index])
  input_label = torch.LongTensor([class_index]*num_samples_total)
else:
  input_label = None


## Define Samples Generator for One Photo and Mosaic Generator

In [18]:
def generate_samples(noise_vector, input_label, instance_vector, num_samples_total, batch_size):
    """Generate GAN outputs and compute feature distances in batches."""
    all_outs, all_dists = [], []
    for i_bs in range(num_samples_total // batch_size + 1):
        start = i_bs * batch_size
        end = min(start + batch_size, num_samples_total)
        if start == end:
            break

        out = get_output(noise_vector[start:end],
                         input_label[start:end] if input_label is not None else None,
                         instance_vector[start:end])

        out_ = preprocess_generated_image(out)
        with torch.no_grad():
            out_features, _ = feature_extractor(out_.cuda())
        out_features /= torch.linalg.norm(out_features, dim=-1, keepdims=True)

        dists = sklearn.metrics.pairwise_distances(
            out_features.cpu(), instance_vector[start:end].cpu(), metric="euclidean", n_jobs=-1)
        all_dists.append(np.diagonal(dists))
        all_outs.append(out.detach().cpu())
        del out

    return torch.cat(all_outs), np.concatenate(all_dists)


def create_mosaic(all_outs, all_dists, num_samples_ranked, size):
    """Select the closest samples and arrange them into a grid mosaic."""
    selected_idxs = np.argsort(all_dists)[:num_samples_ranked]
    grid_size = int(np.sqrt(num_samples_ranked))
    mosaic = np.zeros((3, size * grid_size, size * grid_size))

    row_i, col_i = 0, 0
    for j in selected_idxs:
        mosaic[:, row_i*size:row_i*size+size, col_i*size:col_i*size+size] = all_outs[j]
        if row_i == grid_size - 1:
            row_i = 0
            col_i = col_i + 1 if col_i < grid_size - 1 else 0
        else:
            row_i += 1

    return mosaic

## Batch GAN Image Generator Loop

In [19]:
# --- START THE LOOP ---
for filename in tqdm(image_files, desc="Generating GAN image set for file", unit="file"):
    current_image_path = os.path.join(input_folder, filename)
    base_name = Path(filename).stem
    print(f"\n--- Processing: {base_name} ---")

    if not os.path.exists(current_image_path):
        print(f"Warning: Skipping {filename} as it does not exist.")
        continue

    # 1. Obtain instance features
    input_image_tensor = preprocess_input_image(current_image_path, int(size))
    with torch.no_grad():
        input_features, _ = feature_extractor(input_image_tensor.cuda())
    input_features /= torch.linalg.norm(input_features, dim=-1, keepdims=True)
    instance_vector = torch.tensor(input_features, requires_grad=False, device='cuda').repeat(num_samples_total, 1)

    # 2. Generate samples and compute distances
    all_outs, all_dists = generate_samples(
        noise_vector, input_label, instance_vector, num_samples_total, batch_size)

    # 3. Create mosaic
    mosaic = create_mosaic(all_outs, all_dists, num_samples_ranked, int(size))

    # 4. Save result
    save_name = f"{results_output}/{gen_model}_{base_name}_seed{seed if seed is not None else -1}.png"
    pil_image = save(mosaic[np.newaxis, ...], save_name, torch_format=False)
    print(f"Saved result to: {save_name}")

print("\nAll images processed successfully!")

Generating GAN image set for file:   0%|          | 0/5 [00:00<?, ?file/s]


--- Processing: 000000300121 ---


/tmp/ipykernel_7347/1204634260.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  instance_vector = torch.tensor(input_features, requires_grad=False, device='cuda').repeat(num_samples_total, 1)
/tmp/ipykernel_7347/3433334880.py:38: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  mosaic[:, row_i*size:row_i*size+size, col_i*size:col_i*size+size] = all_outs[j]


Saved result to: /content/icgan_results/icgan_000000300121_seed50.png

--- Processing: 000000232463 ---


/tmp/ipykernel_7347/1204634260.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  instance_vector = torch.tensor(input_features, requires_grad=False, device='cuda').repeat(num_samples_total, 1)
/tmp/ipykernel_7347/3433334880.py:38: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  mosaic[:, row_i*size:row_i*size+size, col_i*size:col_i*size+size] = all_outs[j]


...

Saved result to: /content/icgan_results/icgan_000000131882_seed50.png

--- Processing: 000000378844 ---


/tmp/ipykernel_7347/1204634260.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  instance_vector = torch.tensor(input_features, requires_grad=False, device='cuda').repeat(num_samples_total, 1)
/tmp/ipykernel_7347/3433334880.py:38: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  mosaic[:, row_i*size:row_i*size+size, col_i*size:col_i*size+size] = all_outs[j]


Saved result to: /content/icgan_results/icgan_000000378844_seed50.png

--- Processing: 000000294747 ---


/tmp/ipykernel_7347/1204634260.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  instance_vector = torch.tensor(input_features, requires_grad=False, device='cuda').repeat(num_samples_total, 1)
/tmp/ipykernel_7347/3433334880.py:38: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  mosaic[:, row_i*size:row_i*size+size, col_i*size:col_i*size+size] = all_outs[j]


Saved result to: /content/icgan_results/icgan_000000294747_seed50.png

All images processed successfully!


## Split Generated Images - Batch Processing


In [20]:
import os
from PIL import Image

def split_image(image_path, output_folder, tile_size=256):
    """Splits a single image into tiles."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    base_name = os.path.splitext(os.path.basename(image_path))[0]
    img = Image.open(image_path)
    width, height = img.size

    count = 0
    for top in range(0, height, tile_size):
        for left in range(0, width, tile_size):
            right = min(left + tile_size, width)
            bottom = min(top + tile_size, height)

            tile = img.crop((left, top, right, bottom))

            # Naming format: originalName_tile_0.png
            tile_filename = f"{base_name}_tile_{count}.png"
            tile.save(os.path.join(output_folder, tile_filename))
            count += 1
    return count

input_folder = "/content/icgan_results"  # Folder containing your images
output_root = "/content/all_output_tiles"   # Where tiles will be saved

# 1. Get a list of all images in the folder
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
files_to_process = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_extensions)]

print(f"Found {len(files_to_process)} images. Starting split...")

total_tiles_created = 0

# 2. Loop through the folder and call your function
for filename in files_to_process:
    full_path = os.path.join(input_folder, filename)

    print(f"Processing: {filename}...")
    tiles_created = split_image(full_path, output_root)
    total_tiles_created += tiles_created

print("-" * 30)
print(f"Batch Processing Complete!")
print(f"Total images processed: {len(files_to_process)}")
print(f"Total tiles generated: {total_tiles_created}")

Found 5 images. Starting split...
Processing: icgan_000000232463_seed50.png...
Processing: icgan_000000378844_seed50.png...
Processing: icgan_000000300121_seed50.png...
Processing: icgan_000000131882_seed50.png...
Processing: icgan_000000294747_seed50.png...
------------------------------
Batch Processing Complete!
Total images processed: 5
Total tiles generated: 80


## Export generated GAN Images to Google Drive folder

In [21]:
import shutil
import os

# Where your files are currently (the folder we've been working with)
source_folder = '/content/all_output_tiles'

# Where you want them to go in Google Drive
# Note: 'My Project' is a folder I'm pretending exists in your Drive
destination_folder = '/content/drive/MyDrive/GAN_Generated_Fake_Images'

# Create the destination folder if it doesn't exist yet
os.makedirs(destination_folder, exist_ok=True)

# This copies the entire folder structure
!cp -r "{source_folder}/." "{destination_folder}"
print("Files copied to Drive successfully!")

Files copied to Drive successfully!


## Cleanup Folders

In [ ]:
import os, shutil

def delete_files_in_folder(folder):
  if not os.path.exists(folder):
    return
  for filename in os.listdir(folder):
      file_path = os.path.join(folder, filename)
      try:
          if os.path.isfile(file_path) or os.path.islink(file_path):
              os.unlink(file_path) # Deletes file or symlink
          elif os.path.isdir(file_path):
              shutil.rmtree(file_path) # Deletes subfolder
      except Exception as e:
          print(f'Failed to delete {file_path}. Reason: {e}')

folder = '/content/all_output_tiles'
delete_files_in_folder(folder)
folder = '/content/icgan_results'
delete_files_in_folder(folder)

